In [1]:
%cd ..

/home/dmoreno/pipeline_v4_final/pipeline/training/stamp_classifier/data_acquisition/rubin


In [2]:
import pandas as pd
import numpy as np
import glob

In [3]:
path_stamps = glob.glob('simulated_data/fakes/*.npy')
path_stamps

['simulated_data/fakes/611256378312376542_z_2024120800360_8_592913913020940296_z_2024111700354_4_0.33.npy',
 'simulated_data/fakes/609780833707896898_g_2024112900266_0_650018217640591465_g_2024112600116_4_0.88.npy',
 'simulated_data/fakes/611255691117605475_r_2024113000163_3_614437609048899879_r_2024112600305_1_0.74.npy',
 'simulated_data/fakes/611255622398132035_i_2024120300157_8_648363383921311853_i_2024112300236_6_0.90.npy',
 'simulated_data/fakes/609788117972430021_i_2024111900089_8_650018148921114792_i_2024112600128_5_0.57.npy',
 'simulated_data/fakes/611254454167020410_u_2024113000173_4_614437609048899610_u_2024120100460_7_0.40.npy',
 'simulated_data/fakes/611254454167020410_g_2024113000170_7_648366201419858478_g_2024120700172_2_0.88.npy',
 'simulated_data/fakes/609788942606153550_u_2024113000173_5_609782139377943544_u_2024113000174_7_0.83.npy',
 'simulated_data/fakes/611256447031856188_r_2024110800306_5_609782139377942804_r_2024113000163_8_0.62.npy',
 'simulated_data/fakes/61125

In [4]:
def parse_filename(filepath):
    """
    Extrae los metadatos del nombre de archivo según la convención del README.
    Devuelve un diccionario con los metadatos.
    """
    try:
        # 1. Obtiene solo el nombre del archivo, sin la ruta ni la extensión.
        # e.g., '.../file.npy' -> 'file'
        base_name = os.path.splitext(os.path.basename(filepath))[0]
        
        # 2. Divide el nombre del archivo por el guion bajo '_'
        parts = base_name.split('_')
        
        # 3. Asigna cada parte a su campo correspondiente, convirtiendo a número si es necesario.
        metadata = {
            'gal_objectId': parts[0],
            'gal_band': parts[1],
            'gal_visit': parts[2],
            'gal_detector': int(parts[3]),
            'diff_sourceId': parts[4],
            'diff_band': parts[5],
            'diff_visit': parts[6],
            'diff_detector': int(parts[7]),
            'SN_ellip_dist': float(parts[8]),
        }
        return metadata
    except (IndexError, ValueError) as e:
        print(f"ADVERTENCIA: No se pudo parsear el nombre del archivo: {filepath}. Error: {e}")
        return None

In [5]:
import numpy as np
import os

data_rows = []

# 2. Iteramos sobre cada path en la lista
for path in path_stamps:
    # Primero, extraemos los metadatos del nombre del archivo
    metadata = parse_filename(path)
    
    # Si el parseo falló, saltamos este archivo
    if metadata is None:
        continue

    # Cargamos los datos de imagen del archivo .npy
    loaded_data = np.load(path)
    science_data = loaded_data[:,:,0]
    template_data = loaded_data[:,:,1]
    difference_data = loaded_data[:,:,2]
    
    # Creamos un diccionario para la fila, combinando metadatos y datos de imagen
    row = {
        **metadata,  # Desempaqueta el diccionario de metadatos aquí
        'flux_Science_data': science_data,
        'flux_Difference_data': difference_data,
        'flux_Template_data': template_data,
        'original_path': path
    }
    
    data_rows.append(row)

# 3. Creamos el DataFrame final
df = pd.DataFrame(data_rows)
df

,gal_objectId,gal_band,gal_visit,gal_detector,diff_sourceId,diff_band,diff_visit,diff_detector,SN_ellip_dist,flux_Science_data,flux_Difference_data,flux_Template_data,original_path
0,611256378312376542,z,2024120800360,8,592913913020940296,z,2024111700354,4,0.33,"[[10.982742, -100.364624, -26.70795, 150.41075...","[[32.616245, -31.05905, -51.33175, 83.870514, ...","[[-21.633503, -69.30557, 24.623798, 66.54024, ...",simulated_data/fakes/611256378312376542_z_2024...
1,609780833707896898,g,2024112900266,0,650018217640591465,g,2024112600116,4,0.88,"[[210.51036, 194.16635, 226.60606, 208.87112, ...","[[4.6535196, -7.1084204, 15.935069, -5.0868855...","[[205.85684, 201.27477, 210.67099, 213.95801, ...",simulated_data/fakes/609780833707896898_g_2024...
2,611255691117605475,r,2024113000163,3,614437609048899879,r,2024112600305,1,0.74,"[[-24.263647, 23.363667, -8.025528, 55.17688, ...","[[-10.19281, 19.809084, -1.9615426, 28.060669,...","[[-14.070837, 3.5545833, -6.063985, 27.116209,...",simulated_data/fakes/611255691117605475_r_2024...
3,611255622398132035,i,2024120300157,8,648363383921311853,i,2024112300236,6,0.90,"[[60.298386, -91.77385, 44.987007, -35.799484,...","[[37.04548, -36.984486, 48.095524, -9.511111, ...","[[23.252905, -54.789364, -3.108515, -26.288372...",simulated_data/fakes/611255622398132035_i_2024...
4,609788117972430021,i,2024111900089,8,650018148921114792,i,2024112600128,5,0.57,"[[64.394684, 132.19717, -14.009651, 126.18045,...","[[-16.556696, 32.335075, -49.94117, 25.10089, ...","[[80.95138, 99.86209, 35.93152, 101.07956, 159...",simulated_data/fakes/609788117972430021_i_2024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4494,609788117972430021,r,2024112700181,2,611255003922833171,r,2024120100166,1,0.21,"[[30.381765, 45.401703, 87.44647, 67.1788, 52....","[[2.8186321, 15.799248, 9.838856, 2.4180062, -...","[[27.563133, 29.602457, 77.60762, 64.760796, 7...",simulated_data/fakes/609788117972430021_r_2024...
4495,611253698252783373,y,2024112000234,4,592914119179370901,y,2024111900349,2,1.00,"[[437.34286, 52.55536, -83.57059, -344.83063, ...","[[280.84787, -283.5208, -74.24422, -102.43766,...","[[156.495, 336.07617, -9.32637, -242.39296, -8...",simulated_data/fakes/611253698252783373_y_2024...
4496,611253560813832654,z,2024120900325,6,611254522886495613,z,2024120800404,0,0.30,"[[476.7009, 839.6953, 672.64343, 893.20953, 97...","[[-58.543423, 141.92401, -47.66682, 78.778076,...","[[535.2443, 697.7713, 720.31024, 814.43146, 96...",simulated_data/fakes/611253560813832654_z_2024...
4497,609781589622141692,i,2024120300155,0,648367507089916359,i,2024112800183,3,1.00,"[[-14.031038, 54.82399, -7.4305515, 7.14402, -...","[[5.214042, 29.72688, -21.895369, -33.847218, ...","[[-19.24508, 25.09711, 14.464817, 40.991238, 3...",simulated_data/fakes/609781589622141692_i_2024...


In [6]:
df_sinteticos = df[df.SN_ellip_dist < 0.5].rename(columns={'gal_band': 'band'})
df_sinteticos

,gal_objectId,band,gal_visit,gal_detector,diff_sourceId,diff_band,diff_visit,diff_detector,SN_ellip_dist,flux_Science_data,flux_Difference_data,flux_Template_data,original_path
0,611256378312376542,z,2024120800360,8,592913913020940296,z,2024111700354,4,0.33,"[[10.982742, -100.364624, -26.70795, 150.41075...","[[32.616245, -31.05905, -51.33175, 83.870514, ...","[[-21.633503, -69.30557, 24.623798, 66.54024, ...",simulated_data/fakes/611256378312376542_z_2024...
5,611254454167020410,u,2024113000173,4,614437609048899610,u,2024120100460,7,0.40,"[[49.542225, -8.923589, 75.038475, 37.86905, 4...","[[12.643394, 14.074204, 75.52924, 33.123417, 3...","[[36.89883, -22.997793, -0.4907656, 4.745632, ...",simulated_data/fakes/611254454167020410_u_2024...
9,611254248008597650,y,2024112000222,8,592914119179370901,y,2024111900349,2,0.29,"[[-45.61441, -393.84644, -204.19164, 696.6609,...","[[45.485214, -429.63184, -337.47598, 626.48224...","[[-91.099625, 35.785393, 133.28435, 70.17868, ...",simulated_data/fakes/611254248008597650_y_2024...
10,609780833707896898,g,2024112900266,0,650018973554835776,g,2024112800182,1,0.25,"[[15.538968, -19.373951, 10.069845, -6.252184,...","[[9.218049, -20.889612, -12.756815, -26.5888, ...","[[6.320919, 1.5156616, 22.82666, 20.336617, 9....",simulated_data/fakes/609780833707896898_g_2024...
16,611254454167020410,z,2024120900329,4,611254522886495591,z,2024120800375,0,0.47,"[[-22.330448, 35.378563, -0.542408, 17.049633,...","[[-21.24968, 9.9037285, -36.221664, -5.6041913...","[[-1.0807679, 25.474833, 35.679256, 22.653824,...",simulated_data/fakes/611254454167020410_z_2024...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4489,609782208097430917,y,2024111900099,8,592916112044195962,y,2024112500296,5,0.30,"[[343.73096, 312.34692, 690.9741, 386.07388, 4...","[[-92.50152, -33.328888, 56.65369, -323.34415,...","[[436.23248, 345.6758, 634.32043, 709.418, 546...",simulated_data/fakes/609782208097430917_y_2024...
4492,611255828556561422,i,2024120300157,2,648365651664044133,i,2024112300241,1,0.29,"[[35.547897, -40.608704, 38.428497, 15.815075,...","[[-3.844152, -26.865538, -25.571295, 9.694095,...","[[39.392048, -13.743167, 63.99979, 6.1209807, ...",simulated_data/fakes/611255828556561422_i_2024...
4494,609788117972430021,r,2024112700181,2,611255003922833171,r,2024120100166,1,0.21,"[[30.381765, 45.401703, 87.44647, 67.1788, 52....","[[2.8186321, 15.799248, 9.838856, 2.4180062, -...","[[27.563133, 29.602457, 77.60762, 64.760796, 7...",simulated_data/fakes/609788117972430021_r_2024...
4496,611253560813832654,z,2024120900325,6,611254522886495613,z,2024120800404,0,0.30,"[[476.7009, 839.6953, 672.64343, 893.20953, 97...","[[-58.543423, 141.92401, -47.66682, 78.778076,...","[[535.2443, 697.7713, 720.31024, 814.43146, 96...",simulated_data/fakes/611253560813832654_z_2024...


In [7]:
df_sinteticos.gal_objectId.nunique()

150

In [8]:
df_sinteticos.diff_sourceId.nunique()

247

In [9]:
for index, row in df_sinteticos[['band', 'diff_band']].iterrows():
    if row['band'] != row['diff_band']:
        print(f'Fila {index}: estamos mal - band: {row["band"]}, diff_band: {row["diff_band"]}')

In [10]:
path_real_stamps_v001 = glob.glob('./data/processed/ts_stamps_v0.0.1_comm_4candmax/stamps/*.pkl')

df_real_all_stamps_v001 = []

for path in path_real_stamps_v001:
    df_real_all_stamps_v001.append(pd.read_pickle(path))

df_real_all_stamps_v001 = pd.concat(df_real_all_stamps_v001)
df_real_all_stamps_v001

,oid,measurement_id,flux_Science_data,flux_Science_header,flux_Template_data,flux_Template_header,flux_Difference_data,flux_Difference_header,variance_Science_data,variance_Science_header,...,mask_Template_data,mask_Template_header,mask_Difference_data,mask_Difference_header,class_x,class_y,class,psfFlux,ra,dec
0,169342395216822494,169342395216822494,"[[-29.90286, 20.050756, -24.448586, 37.9608, 1...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[12.407059, 1.2380395, 5.2192917, -2.3418732,...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-42.505054, 13.39961, -36.91224, 33.745564, ...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[749.8122, 781.7552, 750.5761, 789.218, 773.4...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 8.29...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",bogus,bogus,bogus,1794.719971,305.623329,-22.201292
1,169342395222589770,169342395222589770,"[[6.0892243, 41.692726, 20.464373, 38.24141, 5...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[8.858687, 3.949492, 9.812762, 24.611567, 8.0...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-4.155531, 38.25864, 18.45956, 36.588226, -4...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[773.43225, 791.3081, 771.57007, 786.2522, 76...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.07...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",bogus,bogus,bogus,-2347.967529,305.24083,-21.746045
2,169342395228356610,169342395228356610,"[[2.3531318, 2.3519564, 2.3508527, 2.349675, 2...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[8.5377245, 16.045135, 5.262491, -15.194391, ...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[0.18315524, 1.1887197, 2.2256584, 2.8329298,...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[702.817, 702.817, 702.817, 702.817, 702.817,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.89...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",bogus,bogus,bogus,-10584.624023,307.11691,-22.138993
3,169342395236745501,169342395236745501,"[[-13.643683, -9.305476, -35.914104, -7.162728...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[4.5434065, 17.43689, 5.001696, 7.8263507, 12...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-23.46883, -20.267704, -49.50765, -16.440329...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[698.37286, 700.87274, 682.6204, 703.06885, 7...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.13...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",bogus,bogus,bogus,-1972.344849,306.248979,-21.58878
4,169342395237269506,169342395237269506,"[[22.899294, 55.595303, -20.643349, 35.534683,...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-14.037495, -3.1126611, -17.753826, 0.093163...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[27.705294, 59.402344, -28.094788, 29.818535,...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[738.0622, 773.3888, 716.7294, 750.46564, 742...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",...,"[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 5.22...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...",bogus,bogus,bogus,-2180.706299,306.023528,-21.46169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,..

In [11]:
path_real_stamps = glob.glob('./data/processed/ts_stamps_v0.0.2_comm_4candmax/stamps/*.pkl')

df_real_all_stamps = []

for path in path_real_stamps:
    df_real_all_stamps.append(pd.read_pickle(path))

df_real_all_stamps = pd.concat(df_real_all_stamps)
df_real_all_stamps

,oid,measurement_id,flux_Science_data,flux_Science_header,flux_Template_data,flux_Template_header,flux_Difference_data,flux_Difference_header,variance_Science_data,variance_Science_header,variance_Template_data,variance_Template_header,variance_Difference_data,variance_Difference_header,mask_Science_data,mask_Science_header,mask_Template_data,mask_Template_header,mask_Difference_data,mask_Difference_header
0,169342391392665651,169368779404869680,"[[-11.43412, -61.157898, -36.767532, 2.3479912...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[19.970436, 9.456891, 13.70118, 1.8506378, 5....","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-19.709507, -70.71258, -45.61192, -5.388886,...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[1350.7681, 1316.2396, 1328.7379, 1375.7206, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[48.75372, 48.6074, 44.92881, 53.182293, 40.7...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[1406.3743, 1370.4304, 1383.4205, 1432.3733, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.69...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.69...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU..."
1,169342391392665651,169368781154418723,"[[12.617219, 24.45698, 35.47745, 10.133407, 73...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[9.981423, 20.141548, 23.992834, 33.55316, 59...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-17.836555, -11.826263, -7.5156693, -41.9118...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[172.93977, 179.61725, 185.36595, 171.49901, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[13.86152, 13.508177, 13.792195, 13.8862915, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[189.61496, 196.93031, 203.23215, 188.03621, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.58...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -2.02857206888...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.58...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU..."
2,169342391396859977,169342391396859977,"[[20.201748, 8.987029, 20.140661, 5.541897, 2....","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[11.899033, 14.724549, 8.104676, 9.156919, 10...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[9.553257, -0.75593305, 11.6207695, -3.399367...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[113.74172, 110.11547, 114.21119, 107.290985,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[12.621083, 12.107674, 12.1881275, 14.158052,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[128.269, 124.173134, 128.77019, 121.12555, 1...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.33...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2.33...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU..."
3,169342391461347756,169342391461347756,"[[7.650416, 16.770666, 41.501934, 16.376083, 2...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[21.86371, 25.689003, 24.45363, 27.604593, 15...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[-18.728577, -10.020834, 17.58616, -11.041704...","[SIMPLE, BITPIX, NAXIS, NAXIS1, NAXIS2, EXTEND...","[[118.146385, 123.34396, 135.7948, 123.88186, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[13.748417, 16.173056, 14.099533, 15.934614, ...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...","[[133.68181, 139.71996, 153.50385, 140.3026, 1...","[XTENSION, BITPIX, NAXIS, NAXIS1, NAXIS2, PCOU...

In [12]:
df_real_all_stamps.columns

Index(['oid', 'measurement_id', 'flux_Science_data', 'flux_Science_header',
       'flux_Template_data', 'flux_Template_header', 'flux_Difference_data',
       'flux_Difference_header', 'variance_Science_data',
       'variance_Science_header', 'variance_Template_data',
       'variance_Template_header', 'variance_Difference_data',
       'variance_Difference_header', 'mask_Science_data',
       'mask_Science_header', 'mask_Template_data', 'mask_Template_header',
       'mask_Difference_data', 'mask_Difference_header'],
      dtype='object')

In [13]:
params = {
    "dbname": "ztf",
    "user": "readonly_user",
    "host": "quimal-db2.alerce.online",
    "password": "IvV)FhEDw",
    "port": 5432
}

import json
import sqlalchemy as sa
from sqlalchemy import inspect
from sqlalchemy import text

import pandas as pd


engine = sa.create_engine('postgresql+psycopg2://' + params['user'] \
                          + ':' + params['password'] + '@' + params['host'] \
                          + '/' + params['dbname'])

conn = engine.connect()

inspector = inspect(engine)
tables = inspector.get_table_names()
print('Available Tables:\n', tables)

schemas = inspector.get_schema_names()

print("\nSchemas disponibles en la base de datos:")
print(schemas)

tables_rubin_schema = inspector.get_table_names(schema='multisurvey')

LSST_sid = 1

query = '''
SELECT
    *
FROM
    multisurvey.bands
'''
df_bands = pd.read_sql_query(text(query), conn)
dict_mapping = df_bands[df_bands.sid == LSST_sid].set_index('band')['band_name'].to_dict()
dict_mapping

Available Tables:
 []

Schemas disponibles en la base de datos:
['alerce', 'information_schema', 'multisurvey', 'multisurvey_testing', 'public']


{1: 'g', 2: 'r', 3: 'i', 4: 'z', 5: 'y', 6: 'u'}

In [14]:
df_real = pd.read_parquet('./data/processed/ts_stamps_v0.0.2_comm_4candmax/comm_objs_4candmax.parquet')
df_real.band = df_real.band.map(dict_mapping)
df_real

,oid,sid,measurement_id,class,mjd,ra,dec,band,visit,detector,...,shape_flag_parent_source,isDipole,pixelFlags_crCenter,pixelFlags_nodataCenter,pixelFlags_interpolatedCenter,pixelFlags_saturatedCenter,pixelFlags_suspectCenter,pixelFlags_streakCenter,pixelFlags_injectedCenter,pixelFlags_injected_templateCenter
0,169298432645136566,1,169298432645136566,AGN,60924.330414,8.905992,-44.103805,i,2025090500225,91,...,False,True,False,False,False,False,False,False,False,False
1,169298432645136566,1,169342390804939401,AGN,60934.116476,8.906227,-44.103605,g,2025091500059,43,...,False,False,False,False,False,False,False,False,False,False
2,169298432645136566,1,169342391077569156,AGN,60934.11739,8.906245,-44.103744,g,2025091500061,51,...,False,False,False,False,False,False,False,False,False,False
3,169298432645136566,1,169342391901225029,AGN,60934.120075,8.906182,-44.103707,g,2025091500067,86,...,False,False,False,False,False,False,False,False,False,False
4,169298436562092346,1,169298436562092346,AGN,60924.363906,9.721266,-43.055253,i,2025090500254,138,...,False,True,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,169623885986660770,1,169623885986660770,bogus,60998.222662,7.275447,-43.372673,r,2025111800209,181,...,False,False,False,False,False,False,False,False,False,False
6964,169623886050099223,1,169623886050099223,bogus,60998.223125,9.959908,-45.02071,r,2025111800210,46,...,False,False,False,False,False,False,False,False,False,False
6965,169623886051672110,1,169623886051672110,bogus,60998.223125,9.650085,-44.784965,r,2025111800210,49,...,False,False,False,False,False,False,False,False,False,False
6966,169623886096236648,1,169623886096236648,bogus,60998.223125,8.584275,-43.367622,r,2025111800210,134,...,False,False,False,False,False,False,False,False,False,False


In [15]:
# Por ejemplo, ordenar por mjd (fecha) y luego tomar el primer oid
#df_real = df_real.sort_values('mjd').drop_duplicates(subset=['oid'], keep='first')
#df_real

In [16]:
df_real.columns

Index(['oid', 'sid', 'measurement_id', 'class', 'mjd', 'ra', 'dec', 'band',
       'visit', 'detector', 'snr', 'psfFlux', 'psfFluxErr', 'scienceFlux',
       'scienceFluxErr', 'extendedness', 'reliability', 'bboxSize',
       'centroid_flag', 'psfFlux_flag', 'psfFlux_flag_edge',
       'psfFlux_flag_noGoodPixels', 'forced_PsfFlux_flag',
       'forced_PsfFlux_flag_edge', 'forced_PsfFlux_flag_noGoodPixels',
       'shape_flag', 'shape_flag_no_pixels', 'shape_flag_not_contained',
       'shape_flag_parent_source', 'isDipole', 'pixelFlags_crCenter',
       'pixelFlags_nodataCenter', 'pixelFlags_interpolatedCenter',
       'pixelFlags_saturatedCenter', 'pixelFlags_suspectCenter',
       'pixelFlags_streakCenter', 'pixelFlags_injectedCenter',
       'pixelFlags_injected_templateCenter'],
      dtype='object')

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
import os

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
PREVIOUS_PARTITIONS_PATH = "./partitions_trainSN_valSN_synthetic_dup.parquet"
DUPLICATES_FILE_PATH = "./data/processed/ts_stamps_v0.0.2_comm_4candmax/comm_agn_sne_vs_tran_dup_remove.csv" # <--- NUEVO
OUTPUT_PATH = "./partitions_trainSN_mixed_valSN_real_dup.parquet"

# Nombres de columnas
REAL_OID_COL = 'oid'
REAL_CANDID_COL = 'measurement_id'
SYNTHETIC_GAL_ID_COL = 'gal_objectId'
SYNTHETIC_DIFF_ID_COL = 'diff_sourceId'

FINAL_OID_COL = 'oid'
FINAL_CANDID_COL = 'measurement_id'
CLASS_COL = 'class'
GROUP_COL = 'group_id'
BAND_COL = 'band'
STRATIFY_COL = 'stratify_col'

DUP_OID_COL = 'oid'
DUP_OID_KEPT_COL = 'oid_kept'

N_FOLDS = 5
RANDOM_STATE = 42
TEST_SIZE = 0.2 

df_duplicates = pd.read_csv(DUPLICATES_FILE_PATH)

# ==============================================================================
# 3. ESTANDARIZACIÓN Y MANEJO DE DUPLICADOS
# ==============================================================================
print("Estandarizando...")

# --- Datos Reales ---
df_real.rename(columns={REAL_CANDID_COL: FINAL_CANDID_COL}, inplace=True)
df_real[FINAL_OID_COL] = df_real[REAL_OID_COL].astype(str)
df_real[FINAL_CANDID_COL] = df_real[FINAL_CANDID_COL].astype(str)
df_real['dataset_origin'] = 'real'

# Lógica de Duplicados (Aplica a TODAS las clases, incluidas SN)
dup_map = dict(zip(df_duplicates[DUP_OID_COL].astype(str), df_duplicates[DUP_OID_KEPT_COL].astype(str)))
df_real[GROUP_COL] = df_real[FINAL_OID_COL].map(dup_map).fillna(df_real[FINAL_OID_COL])

# --- Datos Sintéticos ---
df_sinteticos[CLASS_COL] = 'SN'
df_sinteticos.rename(columns={SYNTHETIC_GAL_ID_COL: FINAL_OID_COL, SYNTHETIC_DIFF_ID_COL: FINAL_CANDID_COL}, inplace=True)
df_sinteticos[FINAL_OID_COL] = df_sinteticos[FINAL_OID_COL].astype(str)
df_sinteticos[FINAL_CANDID_COL] = df_sinteticos[FINAL_CANDID_COL].astype(str)

if 'original_path' in df_sinteticos.columns:
    df_sinteticos[FINAL_CANDID_COL] = df_sinteticos['original_path'].apply(lambda x: os.path.splitext(os.path.basename(x))[0]).astype(str)
else:
    df_sinteticos[FINAL_CANDID_COL] = [f'FAKE_CANDID_{i}' for i in range(len(df_sinteticos))]

df_sinteticos[GROUP_COL] = df_sinteticos[FINAL_OID_COL]
df_sinteticos['dataset_origin'] = 'synthetic'

# Estratificación
df_real['stratify_col'] = df_real[CLASS_COL].astype(str) + '_' + df_real[BAND_COL].astype(str)
df_sinteticos['stratify_col'] = df_sinteticos[CLASS_COL].astype(str) + '_' + df_sinteticos[BAND_COL].astype(str)
STRATIFY_COL = 'stratify_col'


# ==============================================================================
# 4. LÓGICA DE TEST (Híbrida)
# ==============================================================================
print("\n--- Construyendo Test Set ---")
previous_partitions = pd.read_parquet(PREVIOUS_PARTITIONS_PATH)
# IDs del test anterior (NO-SN)
prev_test_ids = previous_partitions[previous_partitions['partition'] == 'test'][FINAL_CANDID_COL].unique()

# 1. Separar clases
df_real_sn = df_real[df_real[CLASS_COL] == 'SN'].copy()
df_real_non_sn = df_real[df_real[CLASS_COL] != 'SN'].copy()

# 2. Clases NO-SN: Mantener consistencia con IDs previos
#    (Los duplicados ya estaban manejados implícitamente si la partición previa fue correcta)
df_test_non_sn = df_real_non_sn[df_real_non_sn[FINAL_CANDID_COL].isin(prev_test_ids)].copy()
df_train_val_non_sn = df_real_non_sn[~df_real_non_sn[FINAL_CANDID_COL].isin(prev_test_ids)].copy()

# 3. Clase SN (REAL): Hacer nuevo Split 80/20 respetando duplicados
#    Usamos GROUP_COL para asegurar que los duplicados van juntos
unique_sn_groups = df_real_sn[GROUP_COL].unique()

# Estratificación simple por clase (ya que son todas SN) o aleatoria si hay pocos grupos
try:
    groups_train_val_sn, groups_test_sn = train_test_split(
        unique_sn_groups, 
        test_size=TEST_SIZE, 
        random_state=RANDOM_STATE
    )
except ValueError:
    print("Advertencia: Pocos grupos SN para split normal, usando random simple.")
    groups_train_val_sn, groups_test_sn = train_test_split(unique_sn_groups, test_size=TEST_SIZE, random_state=RANDOM_STATE)

df_test_sn = df_real_sn[df_real_sn[GROUP_COL].isin(groups_test_sn)].copy()
df_train_val_sn = df_real_sn[df_real_sn[GROUP_COL].isin(groups_train_val_sn)].copy()

# 4. Unir para TEST final
df_test = pd.concat([df_test_non_sn, df_test_sn], ignore_index=True)
df_test['partition'] = 'test'

# 5. Unir para Train/Val Pool REAL (Ahora incluye SN reales)
df_train_val_real = pd.concat([df_train_val_non_sn, df_train_val_sn], ignore_index=True)

print(f"Test Set: {len(df_test)} filas (SN Reales: {len(df_test_sn)})")
print(f"Train/Val Pool (Real): {len(df_train_val_real)} filas (SN Reales: {len(df_train_val_sn)})")


# ==============================================================================
# 5. K-FOLD (Estrategia Mixta Corregida)
# ==============================================================================
cols_to_save = [FINAL_OID_COL, FINAL_CANDID_COL, CLASS_COL, 'partition', 'dataset_origin']
partitions_list = [df_test[cols_to_save]]

# Datos Sintéticos (Solo para TRAINING)
df_train_synthetic = df_sinteticos.copy()

# K-Fold sobre datos REALES (SN y NO-SN)
X = df_train_val_real.index.to_numpy()
y = df_train_val_real[STRATIFY_COL].to_numpy()
groups = df_train_val_real[GROUP_COL].to_numpy()

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"\nGenerando {N_FOLDS} folds...")

# Intentar estratificar, fallback a solo clase si falla
try:
    splitter = sgkf.split(X, y, groups=groups)
    next(splitter); splitter = sgkf.split(X, y, groups=groups)
except ValueError:
    print("Advertencia: Falló estratificación por clase+banda. Usando solo clase.")
    y = df_train_val_real[CLASS_COL].to_numpy()
    splitter = sgkf.split(X, y, groups=groups)

for fold_idx, (train_idx, val_idx) in enumerate(splitter):
    # 1. Datos Reales (SN y NO-SN)
    train_real = df_train_val_real.iloc[train_idx].copy()
    val_real = df_train_val_real.iloc[val_idx].copy()
    
    # 2. Inyectar Sintéticos (SN) -> SOLO EN TRAINING
    #    Ahora el training tiene: SN Reales + SN Sintéticas + Otras Clases Reales
    train_fold_combined = pd.concat([train_real, df_train_synthetic], ignore_index=True)
    
    # 3. Asignar etiquetas
    train_fold_combined['partition'] = f'training_{fold_idx}'
    val_real['partition'] = f'validation_{fold_idx}'
    
    partitions_list.extend([train_fold_combined[cols_to_save], val_real[cols_to_save]])

final_df = pd.concat(partitions_list, ignore_index=True)

# ==============================================================================
# Celda 6: Guardado y Verificación
# ==============================================================================
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
final_df.to_parquet(OUTPUT_PATH, index=False)
print(f"\nArchivo guardado en: {OUTPUT_PATH}")

print("\n--- Resumen de Particiones ---")
print(final_df.groupby(['partition', 'dataset_origin']).size())

print("\n--- Verificaciones ---")
# Verificar que validación solo tenga reales
val_synthetic = final_df[
    (final_df['partition'].str.startswith('validation')) & 
    (final_df['dataset_origin'] == 'synthetic')
]
if len(val_synthetic) == 0:
    print("EXITO: No hay datos sintéticos en validación.")
else:
    print(f"ERROR: Se encontraron {len(val_synthetic)} datos sintéticos en validación.")

# Verificar fuga de OIDs reales entre train y val (Fold 0)
train_0_oids = set(final_df[(final_df['partition']=='training_0') & (final_df['dataset_origin']=='real')][FINAL_OID_COL])
val_0_oids = set(final_df[(final_df['partition']=='validation_0')][FINAL_OID_COL])
leak = train_0_oids.intersection(val_0_oids)
if len(leak) == 0:
    print("EXITO: No hay fuga de objetos reales entre Train_0 y Val_0.")
else:
    print(f"ERROR: Fuga detectada en {len(leak)} objetos.")

Estandarizando...

--- Construyendo Test Set ---
Test Set: 1356 filas (SN Reales: 35)
Train/Val Pool (Real): 5612 filas (SN Reales: 153)

Generando 5 folds...


/home/dmoreno/miniconda3/envs/stamp_rubin/lib/python3.10/site-packages/sklearn/model_selection/_split.py:1035: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/home/dmoreno/miniconda3/envs/stamp_rubin/lib/python3.10/site-packages/sklearn/model_selection/_split.py:1035: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



Archivo guardado en: ./partitions_trainSN_mixed_valSN_real_dup.parquet

--- Resumen de Particiones ---
partition     dataset_origin
test          real              1356
training_0    real              4470
              synthetic         1750
training_1    real              4509
              synthetic         1750
training_2    real              4520
              synthetic         1750
training_3    real              4441
              synthetic         1750
training_4    real              4508
              synthetic         1750
validation_0  real              1142
validation_1  real              1103
validation_2  real              1092
validation_3  real              1171
validation_4  real              1104
dtype: int64

--- Verificaciones ---
EXITO: No hay datos sintéticos en validación.
EXITO: No hay fuga de objetos reales entre Train_0 y Val_0.


In [24]:
OUTPUT_PATH

'./partitions_trainSN_mixed_valSN_real_dup.parquet'

In [25]:
import pandas as pd

partitions_poc = pd.read_parquet(f'{OUTPUT_PATH}')
#partitions_poc = pd.read_parquet(f'partitions_trainSN_mixed_valSN_real.parquet')
display(partitions_poc)

# Asumiendo que tu DataFrame se llama 'df'
conteo = partitions_poc.groupby(['partition', 'class']).size().reset_index(name='count')

# Para una visualización más clara, puedes pivotar la tabla
conteo_pivot = conteo.pivot(index='partition', columns='class', values='count').fillna(0)

print(conteo_pivot)

,oid,measurement_id,class,partition,dataset_origin
0,169302853145854613,169302853145854613,AGN,test,real
1,169302853145854613,169342391323459848,AGN,test,real
2,169302853145854613,169342391877108661,AGN,test,real
3,169302853145854613,169368781252985128,AGN,test,real
4,169302854308200461,169302854308200461,AGN,test,real
...,...,...,...,...,...
38161,169368775131398252,169368775131398252,SN,validation_4,real
38162,169368775131398252,169368781696532571,SN,validation_4,real
38163,169368775131398252,169368790533406737,SN,validation_4,real
38164,169368776457846868,169368776457846868,SN,validation_4,real


class         AGN    SN   VS  asteroid  bogus
partition                                    
test          247    35  218       432    424
training_0    815  1872  664      1468   1401
training_1    861  1876  653      1461   1408
training_2    884  1870  689      1462   1365
training_3    815  1860  643      1472   1401
training_4    869  1884  671      1441   1393
validation_0  246    31  166       358    341
validation_1  200    27  177       365    334
validation_2  177    33  141       364    377
validation_3  246    43  187       354    341
validation_4  192    19  159       385    349


In [26]:
partitions_poc.dtypes

oid               object
measurement_id    object
class             object
partition         object
dataset_origin    object
dtype: object

In [27]:
synthetic = partitions_poc[partitions_poc['dataset_origin'] == 'synthetic']
synthetic

,oid,measurement_id,class,partition,dataset_origin
5826,611256378312376542,611256378312376542_z_2024120800360_8_592913913...,SN,training_0,synthetic
5827,611254454167020410,611254454167020410_u_2024113000173_4_614437609...,SN,training_0,synthetic
5828,611254248008597650,611254248008597650_y_2024112000222_8_592914119...,SN,training_0,synthetic
5829,609780833707896898,609780833707896898_g_2024112900266_0_650018973...,SN,training_0,synthetic
5830,611254454167020410,611254454167020410_z_2024120900329_4_611254522...,SN,training_0,synthetic
...,...,...,...,...,...
37057,609782208097430917,609782208097430917_y_2024111900099_8_592916112...,SN,training_4,synthetic
37058,611255828556561422,611255828556561422_i_2024120300157_2_648365651...,SN,training_4,synthetic
37059,609788117972430021,609788117972430021_r_2024112700181_2_611255003...,SN,training_4,synthetic
37060,611253560813832654,611253560813832654_z_2024120900325_6_611254522...,SN,training_4,synthetic


In [28]:
training_0 = synthetic[synthetic['partition'] == 'training_0']
training_0

,oid,measurement_id,class,partition,dataset_origin
5826,611256378312376542,611256378312376542_z_2024120800360_8_592913913...,SN,training_0,synthetic
5827,611254454167020410,611254454167020410_u_2024113000173_4_614437609...,SN,training_0,synthetic
5828,611254248008597650,611254248008597650_y_2024112000222_8_592914119...,SN,training_0,synthetic
5829,609780833707896898,609780833707896898_g_2024112900266_0_650018973...,SN,training_0,synthetic
5830,611254454167020410,611254454167020410_z_2024120900329_4_611254522...,SN,training_0,synthetic
...,...,...,...,...,...
7571,609782208097430917,609782208097430917_y_2024111900099_8_592916112...,SN,training_0,synthetic
7572,611255828556561422,611255828556561422_i_2024120300157_2_648365651...,SN,training_0,synthetic
7573,609788117972430021,609788117972430021_r_2024112700181_2_611255003...,SN,training_0,synthetic
7574,611253560813832654,611253560813832654_z_2024120900325_6_611254522...,SN,training_0,synthetic


In [29]:
training_0.oid.nunique()

150

In [30]:
df_real

,oid,sid,measurement_id,class,mjd,ra,dec,band,visit,detector,...,pixelFlags_nodataCenter,pixelFlags_interpolatedCenter,pixelFlags_saturatedCenter,pixelFlags_suspectCenter,pixelFlags_streakCenter,pixelFlags_injectedCenter,pixelFlags_injected_templateCenter,dataset_origin,group_id,stratify_col
0,169298432645136566,1,169298432645136566,AGN,60924.330414,8.905992,-44.103805,i,2025090500225,91,...,False,False,False,False,False,False,False,real,169298432645136566,AGN_i
1,169298432645136566,1,169342390804939401,AGN,60934.116476,8.906227,-44.103605,g,2025091500059,43,...,False,False,False,False,False,False,False,real,169298432645136566,AGN_g
2,169298432645136566,1,169342391077569156,AGN,60934.11739,8.906245,-44.103744,g,2025091500061,51,...,False,False,False,False,False,False,False,real,169298432645136566,AGN_g
3,169298432645136566,1,169342391901225029,AGN,60934.120075,8.906182,-44.103707,g,2025091500067,86,...,False,False,False,False,False,False,False,real,169298432645136566,AGN_g
4,169298436562092346,1,169298436562092346,AGN,60924.363906,9.721266,-43.055253,i,2025090500254,138,...,False,False,False,False,False,False,False,real,169298436562092346,AGN_i
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6963,169623885986660770,1,169623885986660770,bogus,60998.222662,7.275447,-43.372673,r,2025111800209,181,...,False,False,False,False,False,False,False,real,169623885986660770,bogus_r
6964,169623886050099223,1,169623886050099223,bogus,60998.223125,9.959908,-45.02071,r,2025111800210,46,...,False,False,False,False,False,False,False,real,169623886050099223,bogus_r
6965,169623886051672110,1,169623886051672110,bogus,60998.223125,9.650085,-44.784965,r,2025111800210,49,...,False,False,False,False,False,False,False,real,169623886051672110,bogus_r
6966,169623886096236648,1,169623886096236648,bogus,60998.223125,8.584275,-43.367622,r,2025111800210,134,...,False,False,False,False,False,False,False,real,169623886096236648,bogus_r


In [31]:
partitions_poc

,oid,measurement_id,class,partition,dataset_origin
0,169302853145854613,169302853145854613,AGN,test,real
1,169302853145854613,169342391323459848,AGN,test,real
2,169302853145854613,169342391877108661,AGN,test,real
3,169302853145854613,169368781252985128,AGN,test,real
4,169302854308200461,169302854308200461,AGN,test,real
...,...,...,...,...,...
38161,169368775131398252,169368775131398252,SN,validation_4,real
38162,169368775131398252,169368781696532571,SN,validation_4,real
38163,169368775131398252,169368790533406737,SN,validation_4,real
38164,169368776457846868,169368776457846868,SN,validation_4,real


In [33]:
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord
from astropy import units as u

In [35]:
print("Datos cargados.")

# ==============================================================================
# 2. UNIR COORDENADAS CON PARTICIONES
# ==============================================================================
# Asegúrate de que la columna de ID se llame igual ('oid')
# Filtramos para tener solo una fila por objeto único (ya que las coords son por objeto)
df_analysis = pd.merge(partitions_poc, df_real, on='oid', how='inner')
# Si partitions tiene filas por detección, nos quedamos con una por objeto
df_analysis = df_analysis.drop_duplicates(subset=['oid']) 

print(f"Total de objetos únicos analizados: {len(df_analysis)}")

# ==============================================================================
# 3. FUNCIÓN DE ANÁLISIS DE VECINOS
# ==============================================================================
def analizar_vecinos_en_grupo(df_grupo, nombre_grupo):
    if len(df_grupo) < 2:
        return None
    
    # 1. Asegurar que RA y DEC sean numéricos (floats) y convertirlos a numpy arrays puros
    # Esto elimina cualquier metadata de PyArrow/Pandas que pueda causar conflicto
    ra_values = df_grupo['ra'].astype(float).to_numpy()
    dec_values = df_grupo['dec'].astype(float).to_numpy()
        
    # 2. Crear SkyCoord usando los arrays de numpy puros
    coords = SkyCoord(ra=ra_values * u.degree, 
                      dec=dec_values * u.degree)
    
    # Buscar 2do vecino más cercano
    idx, d2d, _ = coords.match_to_catalog_sky(coords, nthneighbor=2)
    
    # Crear resultado
    resultados = pd.DataFrame({
        'partition': nombre_grupo,
        'oid_1': df_grupo['oid'].values,
        'oid_2': df_grupo.iloc[idx]['oid'].values,
        'separation_arcsec': d2d.arcsec
    })
    
    # Ordenar y tomar el mínimo
    min_row = resultados.sort_values('separation_arcsec').iloc[0]
    return min_row

# ==============================================================================
# 4. EJECUCIÓN POR PARTICIÓN
# ==============================================================================
print("\n--- Resultados: Separación mínima por partición ---")

resultados_globales = []

# Iterar sobre cada partición única (training_0, validation_0, test, etc.)
for part_name in df_analysis['partition'].unique():
    # Filtrar datos de esa partición
    df_part = df_analysis[df_analysis['partition'] == part_name]
    
    # Calcular
    res = analizar_vecinos_en_grupo(df_part, part_name)
    
    if res is not None:
        resultados_globales.append(res)
        print(f"[{part_name}] Mínima separación: {res['separation_arcsec']:.6f} arcsec "
              f"(Entre {res['oid_1']} y {res['oid_2']})")

# ==============================================================================
# 5. RESUMEN FINAL
# ==============================================================================
if resultados_globales:
    df_final_results = pd.DataFrame(resultados_globales).sort_values('separation_arcsec')
    
    print("\n" + "="*60)
    print("RESUMEN GLOBAL (Ordenado por cercanía):")
    print(df_final_results)
    print("="*60)
    
    # Alerta crítica
    limite_alerta = 1.0 # arcsec
    muy_cercanos = df_final_results[df_final_results['separation_arcsec'] < limite_alerta]
    
    if not muy_cercanos.empty:
        print(f"\n⚠️ ADVERTENCIA: Se encontraron {len(muy_cercanos)} pares con separación < {limite_alerta}\" dentro de la misma partición.")
        print("Esto sugiere duplicados no resueltos que podrían afectar el entrenamiento/evaluación.")
    else:
        print(f"\n✅ Todo bien: No hay objetos extremadamente cercanos (< {limite_alerta}\") dentro de las particiones.")

Datos cargados.
Total de objetos únicos analizados: 4299

--- Resultados: Separación mínima por partición ---
[test] Mínima separación: 0.005081 arcsec (Entre 169562329369280585 y 169562329523421274)
[training_0] Mínima separación: 0.003061 arcsec (Entre 169342390926050043 y 169342391330275367)
[validation_0] Mínima separación: 0.003163 arcsec (Entre 169302853752455193 y 169302853103910941)

RESUMEN GLOBAL (Ordenado por cercanía):
        partition               oid_1               oid_2  separation_arcsec
79     training_0  169342390926050043  169342391330275367           0.003061
104  validation_0  169302853752455193  169302853103910941           0.003163
236          test  169562329369280585  169562329523421274           0.005081

⚠️ ADVERTENCIA: Se encontraron 3 pares con separación < 1.0" dentro de la misma partición.
Esto sugiere duplicados no resueltos que podrían afectar el entrenamiento/evaluación.


In [36]:
import pandas as pd
import numpy as np
from astropy.coordinates import SkyCoord, match_coordinates_sky
from astropy import units as u
import itertools

# ==============================================================================
# 1. CARGAR DATOS Y PREPARAR
# ==============================================================================
# ... (Asumiendo que df_analysis ya tiene 'oid', 'ra', 'dec', 'partition' cargados como antes) ...

print(f"Total de objetos únicos para analizar: {len(df_analysis)}")

# Función auxiliar para obtener coords de una partición
def get_coords(df, partition_name):
    subset = df[df['partition'] == partition_name]
    coords = SkyCoord(ra=subset['ra'].astype(float).to_numpy()*u.degree, 
                      dec=subset['dec'].astype(float).to_numpy()*u.degree)
    return coords, subset['oid'].to_numpy()

# ==============================================================================
# 2. CÁLCULO DE SEPARACIÓN ENTRE CONJUNTOS
# ==============================================================================
print("\n--- Análisis de Separación ENTRE Conjuntos (Busca fugas) ---")

# Definir los pares a comparar. 
# Queremos comparar cada fold de entrenamiento con su validación, y todo con test.
# Pero para simplificar y ver lo peor, compararemos todas las combinaciones posibles de particiones.

partition_names = df_analysis['partition'].unique()
# Generar todas las combinaciones de 2 particiones distintas
pairs = list(itertools.combinations(partition_names, 2))

resultados_cross = []

for part1, part2 in pairs:
    # Optimización: Solo nos interesa comparar Train vs Val del MISMO fold, o Train vs Test.
    # No tiene sentido comparar Training_0 con Validation_1, por ejemplo.
    
    es_train_val_mismo_fold = (part1.startswith('training_') and part2.replace('validation', 'training') == part1) or \
                              (part2.startswith('training_') and part1.replace('validation', 'training') == part2)
    
    es_con_test = ('test' in part1) or ('test' in part2)
    
    # Solo procesamos si es una comparación relevante para fuga de datos
    if es_train_val_mismo_fold or es_con_test:
        
        print(f"Comparando {part1} vs {part2}...")
        
        coords1, oids1 = get_coords(df_analysis, part1)
        coords2, oids2 = get_coords(df_analysis, part2)
        
        if len(coords1) == 0 or len(coords2) == 0:
            continue
            
        # match_coordinates_sky encuentra para cada objeto en coords1, el más cercano en coords2
        idx, d2d, _ = match_coordinates_sky(coords1, coords2)
        
        # Encontrar la mínima distancia en este cruce
        min_dist_idx = np.argmin(d2d)
        min_dist_arcsec = d2d[min_dist_idx].arcsec
        
        resultados_cross.append({
            'set_1': part1,
            'set_2': part2,
            'min_separation_arcsec': min_dist_arcsec,
            'oid_1': oids1[min_dist_idx],
            'oid_2': oids2[idx[min_dist_idx]]
        })

# ==============================================================================
# 3. RESULTADOS
# ==============================================================================
if resultados_cross:
    df_cross_results = pd.DataFrame(resultados_cross).sort_values('min_separation_arcsec')
    
    print("\n" + "="*80)
    print("RESUMEN DE SEPARACIÓN MÍNIMA ENTRE CONJUNTOS (Train vs Val / Train vs Test):")
    print(df_cross_results)
    print("="*80)
    
    # Alerta de Fuga
    limite_critico = 1.0 # arcsec (Si es menor a esto, probablemente es el mismo objeto)
    fugas = df_cross_results[df_cross_results['min_separation_arcsec'] < limite_critico]
    
    if not fugas.empty:
        print(f"\n🚨 ALERTA ROJA: Se encontraron {len(fugas)} pares de conjuntos con objetos a < {limite_critico}\".")
        print("Esto indica FUGA DE DATOS (Data Leakage). El mismo objeto físico está en ambos sets con distinto ID.")
        print(fugas)
    else:
        print(f"\n✅ EXCELENTE: La separación mínima entre conjuntos es > {limite_critico}\". No hay evidencia espacial de fuga de datos.")

Total de objetos únicos para analizar: 4299

--- Análisis de Separación ENTRE Conjuntos (Busca fugas) ---
Comparando test vs training_0...
Comparando test vs validation_0...
Comparando training_0 vs validation_0...

RESUMEN DE SEPARACIÓN MÍNIMA ENTRE CONJUNTOS (Train vs Val / Train vs Test):
        set_1         set_2  min_separation_arcsec               oid_1  \
2  training_0  validation_0               0.150800  169342391071801790   
0        test    training_0               1.249692  169342391509582427   
1        test  validation_0              14.087339  169342390827483268   

                oid_2  
2  169342390936011001  
0  169355602395398209  
1  169615082866081802  

🚨 ALERTA ROJA: Se encontraron 1 pares de conjuntos con objetos a < 1.0".
Esto indica FUGA DE DATOS (Data Leakage). El mismo objeto físico está en ambos sets con distinto ID.
        set_1         set_2  min_separation_arcsec               oid_1  \
2  training_0  validation_0                 0.1508  1693423910718